# Logistic Regression
## class_weights

In [21]:
# ---- class_weight comparison: does weighting the minority class reduce false negatives? ----
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, classification_report, recall_score
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [22]:
# Rebuild X, y (Amyloid + Total tau, MCI only) — same as cell 45
df = pd.read_csv("data/plasma_lipidomics.csv")

In [23]:
mci['target'] = (mci["Progression to Alzheimer's Disease"] == 'Yes').astype(int)
y = mci['target']
X = mci[['CSF Amyloid (pg/mL)', 'CSF Total tau (pg/mL)']]

print("Class balance (0=No progression, 1=Progressed):")
print(y.value_counts(), "\n")

Class balance (0=No progression, 1=Progressed):
target
1    47
0    42
Name: count, dtype: int64 



In [24]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, permutation_test_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

biomarker_cols = ['CSF Amyloid (pg/mL)', 'CSF Total tau (pg/mL)', 'CSF Phosphorylated tau (pg/mL)']
for col in biomarker_cols:
    df[col] = df.groupby('Diagnostic')[col].transform(lambda s: s.fillna(s.median()))
df['APOE4'] = df.groupby('Diagnostic')['APOE4'].transform(lambda s: s.fillna(s.mode()[0]))
df['APOE4_bin'] = (df['APOE4'] == 'Yes').astype(int)
df['Sex_bin'] = (df['Sex'] == 'Male').astype(int)

amyloid_median = df['CSF Amyloid (pg/mL)'].median()
df['CSF Amyloid (pg/mL)'] = df['CSF Amyloid (pg/mL)'].fillna(amyloid_median)

tau_median = df['CSF Total tau (pg/mL)'].median()
df['CSF Total tau (pg/mL)'] = df['CSF Total tau (pg/mL)'].fillna(tau_median)

df = df[df.Diagnostic == 'Mild Cognitive Impairment'].copy()
df['target'] = (df["Progression to Alzheimer's Disease"] == 'Yes').astype(int)
y = df['target']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ---- Every feature (or combo) you've tested, run through the same permutation test ----
feature_sets = {
    'Age':                 ['Age'],
    'Sex':                 ['Sex_bin'],
    'MMSE':                ['MMSE'],
    'APOE4':               ['APOE4_bin'],
    'CSF Amyloid':         ['CSF Amyloid (pg/mL)'],
    'CSF Total tau':       ['CSF Total tau (pg/mL)'],
    'Amyloid + Total tau': ['CSF Amyloid (pg/mL)', 'CSF Total tau (pg/mL)'],
}


In [10]:
len(df)

89

In [14]:
df.columns

Index(['Sample', 'Diagnostic', 'Sex', 'Age', 'MMSE', 'CSF Amyloid (pg/mL)',
       'CSF Total tau (pg/mL)', 'CSF Phosphorylated tau (pg/mL)', 'APOE4',
       'Progression to Alzheimer's Disease', 'Progression time (months)',
       'APOE4_bin', 'Sex_bin', 'target'],
      dtype='str')

In [25]:

X = df.drop(columns=['Sample', 'Diagnostic', "Progression to Alzheimer's Disease"])

# Preoprocess

In [28]:
import pandas as pd

# 1. Load data, keep only patients with a known outcome (MCI patients)
df = pd.read_csv('data/plasma_lipidomics.csv')
mci = df[df["Progression to Alzheimer's Disease"].notna()].copy()

# 2. Fill missing numeric values with the column median
numeric_cols = ['Age', 'MMSE', 'CSF Amyloid (pg/mL)', 'CSF Total tau (pg/mL)',
                 'CSF Phosphorylated tau (pg/mL)']
for col in numeric_cols:
    median_val = mci[col].median()
    mci[col] = mci[col].fillna(median_val)

# 3. Fill missing categorical values with the most common value
mode_val = mci['APOE4'].mode()[0]
mci['APOE4'] = mci['APOE4'].fillna(mode_val)

# 4. Convert categorical text columns to numeric (0/1)
mci['Sex'] = (mci['Sex'] == 'Male').astype(int)          # Male=1, Female=0
mci['APOE4'] = (mci['APOE4'] == 'Yes').astype(int)        # carries APOE4 allele=1, no=0
mci['Target'] = (mci["Progression to Alzheimer's Disease"] == 'Yes').astype(int)

# 5. Final feature set + target
feature_cols = numeric_cols + ['Sex', 'APOE4']
X = mci[feature_cols]
y = mci['Target']

,Age,MMSE,CSF Amyloid (pg/mL),CSF Total tau (pg/mL),CSF Phosphorylated tau (pg/mL),Sex,APOE4
64,69,23,595.0,465.0,75.0,1,0
104,70,27,1845.0,353.0,92.4,1,0
105,73,29,928.0,531.0,176.0,0,0
106,68,23,619.0,477.0,142.0,0,1
107,78,27,784.0,1231.0,394.0,1,0
...,...,...,...,...,...,...,...
187,75,25,465.0,67.1,22.4,1,0
188,70,27,965.0,408.0,52.7,1,1
189,77,22,604.0,332.0,63.5,0,0
190,77,22,314.0,692.0,84.4,0,1


In [30]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [33]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [2, 3, 4]
}

gbm = GradientBoostingClassifier(random_state=42)

grid_search = GridSearchCV(gbm, param_grid, cv=5, scoring='recall', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV recall: {grid_search.best_score_:.3f}")

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['No progression', 'Progression']))

ValueError: Invalid parameter 'learning_rate' for estimator LogisticRegression(random_state=42). Valid parameters are: ['C', 'class_weight', 'dual', 'fit_intercept', 'intercept_scaling', 'l1_ratio', 'max_iter', 'n_jobs', 'penalty', 'random_state', 'solver', 'tol', 'verbose', 'warm_start'].

In [32]:
final_model = GradientBoostingClassifier(
    learning_rate=0.05,
    max_depth=4,
    n_estimators=200,
    random_state=42
)
final_model.fit(X_train, y_train)

,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.05
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",200
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",4
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (i